In [ ]:
!pip install transformers

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!unzip '*.zip'

In [ ]:
import json

with open("f_messages.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dialogues = []   # 각 conversation 단위 대화 모음
labels = []      # 각 대화에 대한 라벨 (0/1)

for convo in data:
    convo_id = convo["conversation_id"]
    messages = convo["messages"]

    # 모든 메시지를 'Speaker: text' 형태로 묶기
    convo_text = " [SEP] ".join([f"{msg['author']}: {msg['text']}" for msg in messages])

    # 그루밍 의도가 하나라도 있으면 1, 아니면 0
    label = 1 if any(msg["is_offender"] for msg in messages) else 0

    dialogues.append(convo_text)
    labels.append(label)

# 출력 예시
print("Number of conversations:", len(dialogues))
print("Sample conversation:\n", dialogues[0])
print("Label:", labels[0])

In [ ]:
import pandas as pd

df = pd.DataFrame({"text": dialogues, "label": labels})
df.head()

In [ ]:
converted_texts = []

for text in df['text']:
    utterances = text.split(" [SEP] ")
    user_ids = []
    for utt in utterances:
        if ": " in utt:
            user_id = utt.split(": ")[0]
            user_ids.append(user_id)
    unique_users = list(dict.fromkeys(user_ids))
    user_map = {unique_users[0]: 0, unique_users[1]: 1}

    converted_utterances = []
    for utt in utterances:
        if ": " in utt:
            user_id, utterance_text = utt.split(": ", 1)
            new_user_id = str(user_map[user_id])
            converted_utterances.append(f"{new_user_id}: {utterance_text}")
        else:
            converted_utterances.append(utt)

    converted_text = " [SEP] ".join(converted_utterances)
    converted_texts.append(converted_text)


In [ ]:
print(converted_texts)

In [ ]:
df = pd.DataFrame({"text": converted_texts, "label": labels})
df.head()

In [ ]:
df.to_csv("labeling.csv", index=False)